# 6.3.1 Agent Architectures

Simulate a ReAct-style (Reason + Act) agent loop with tool dispatching and trajectory logging.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)

def tool_search(q):
    db = {'weather': 'Sunny 22°C', 'capital france': 'Paris', 'pi': '3.14159'}
    for k, v in db.items():
        if k in q.lower(): return v
    return 'No result'

def tool_calc(expr):
    try: return str(eval(expr, {'__builtins__': {}}, {}))
    except: return 'error'

TOOLS = {'search': tool_search, 'calc': tool_calc}

plan = [
    ('think', 'I need the capital of France'),
    ('act:search', 'capital france'),
    ('observe', None),
    ('think', 'Got it. Now compute 7*6'),
    ('act:calc', '7*6'),
    ('observe', None),
]

trajectory = []; last_result = None
for step, arg in plan:
    if step.startswith('act:'):
        last_result = TOOLS[step.split(':')[1]](arg)
        trajectory.append({'step': step, 'arg': arg, 'result': last_result})
    elif step == 'observe':
        trajectory.append({'step': step, 'arg': last_result, 'result': None})
    else:
        trajectory.append({'step': step, 'arg': arg, 'result': None})

for i, t in enumerate(trajectory):
    print(f'Step {i+1} [{t["step"]}]: {t["arg"]}' + (f' → {t["result"]}' if t['result'] else ''))

In [ ]:
max_steps = list(range(1, 10))
success = [0.1, 0.3, 0.55, 0.72, 0.83, 0.89, 0.92, 0.94, 0.95]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(max_steps, success, 'o-', color='steelblue', lw=2)
ax.set(xlabel='Max Agent Steps', ylabel='Task Success Rate',
       title='ReAct Agent: Performance vs Max Steps')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/agent_architectures.png')
print('Saved agent_architectures.png')